# SRD Pose Emergency Core 코드리뷰 정리

대상 파일: `srd_pose_emergency_core.py`

이번 노트북은 Pose Core 한 파일만 기준으로 정리했다. nav, stt, 관제노드 설명은 제외하고 **코어가 프레임 1장을 받아 어떤 순서로 판단을 내리는지**에 집중한다.

## 1. 한 줄 요약

이 코어는 **OpenCV BGR frame -> pose 추론 -> observation -> posture -> motion -> trapped -> emergency_level** 순서로 처리한 뒤, `annotated frame`과 `structured result list`를 반환하는 분석 엔진이다.

## 2. 전체 흐름

```text
OpenCV BGR frame
  -> YOLO Pose 추론
  -> 유효 keypoint 필터링
  -> observation 판정
  -> posture 판정
  -> motion 계산 / 분류
  -> trapped 보조 판정
  -> state duration 계산
  -> emergency_level 결정
  -> annotated frame + result list 반환
```

## 3. 코드 동작 설명

### 입력
- `frame`: `np.ndarray`
- 형식: OpenCV 기준 BGR image

### 출력
- `annotated`: `np.ndarray`
- `results`: `List[dict]`
- `results_to_json(results)`: `str`
- `extract_frame_emergency_level(results)`: `str | None`

### 중요
Pose Core 내부에는 topic, service, action이 없다. 외부 노드가 Python API를 직접 호출하는 구조다.

## 4. 핵심 함수

| 함수 | 역할 |
|---|---|
| `AnalyzerConfig` | 임계값, 모델 경로, 시각화 옵션을 모아둔 설정 클래스 |
| `_classify_visibility` | FULL_BODY / UPPER_BODY / PARTIAL / LOW_CONF 판정 |
| `_classify_posture` | shoulder_tilt, head_drop_ratio, torso_angle 기반 자세 판정 |
| `_motion_value`, `_classify_motion` | 프레임 간 keypoint 이동량으로 움직임 계산 |
| `_possible_trapped` | 매몰 의심 여부 계산 |
| `_state_duration` | track_id별 관측 시간, 상태 지속 시간 계산 |
| `_decide` | 최종 emergency level 결정 |
| `_pack_result` | structured result dict 생성 |
| `analyze_frame_with_results` | 메인 진입점 |

## 5. 판정 로직 요약

### Observation
유효 keypoint 수와 상/하체 존재 여부로 `LOW_CONF`, `FULL_BODY`, `UPPER_BODY`, `PARTIAL`을 나눈다.

### Posture
- 전신: `shoulder_tilt`, `head_drop_ratio`, `torso_angle`, `aspect`를 함께 사용
- 상반신: `shoulder_tilt`, `head_drop_ratio` 중심으로 사용

### Motion
이전 프레임과 현재 프레임의 keypoint 차이로 움직임을 정량화하고, `motion_window`로 smoothing 한다.

### Emergency Level
`observation + posture + motion + trapped + state_sec` 조합으로 최종 등급을 만든다.

## 6. 결과 dict 스키마

- `track_id`
- `bbox`
- `observation`
- `posture`
- `motion`
- `emergency_level`
- `trapped`
- `seen_sec`, `state_sec`
- `shoulder_tilt`, `head_drop_ratio`, `torso_angle`
- `motion_smooth`, `motion_upper`, `motion_core`
- `rep_point_px`, `rep_point_method` 등 대표점 필드

## 7. 코드 주석 기준

Python에서는 **싱글쿼트(`'`)로 주석을 쓰지 않는다.**

- 설명 주석: `#`
- 함수/클래스 설명: `"""docstring"""`

코드리뷰에서는 이 기준으로 정리하는 것이 맞다.

In [ ]:
# v0.000
# file: comment_guideline_example.py
# date: 2026-03-11
# changes:
# - 코드리뷰용 주석 예시

# 1) observation 판정
# 2) posture 판정
# 3) motion 계산

def analyze_frame_with_results(frame):
    """프레임 1장을 분석해서 annotated와 result list를 반환한다."""
    pass


## 8. 코드리뷰에서 꼭 말할 것

1. Pose Core는 ROS 노드가 아니라 라이브러리다.
2. 입력은 BGR frame 1장, 출력은 annotated frame과 structured result다.
3. observation이 뒤 로직의 큰 분기 기준이다.
4. posture와 motion은 단발 수치가 아니라 임계값 + smoothing + duration을 함께 본다.
5. 최종 emergency level은 `state_sec`까지 포함한 시간 누적형 판정이다.

### 🧠 Vision Node & Core Logic Flowchart

이 셀이 일반 마크다운 글로 보인다면 깃허브(web)에서 보시거나, VSCode에서 `Markdown Preview Mermaid Support` 확장을 설치하시면 다이어그램으로 자동 변환되어 보입니다!

```mermaid
flowchart TD
    %% ROS2 Node External I/O
    subgraph "ROS2 Network (RescueVisionNode)"
        SUB[Subscribe: /camera/image_raw/compressed]
        PUB1[Publish: /robot6/emergency_level]
        PUB2[Publish: /robot6/image_result/compressed]
    end

    %% Decoding & Invocation
    SUB --> DEC[Decode JPG to BGR Frame]
    DEC --> ENGINE[[Call PoseEmergencyEngine.analyze_frame()]]

    %% Core Engine Logic
    subgraph "Core Engine (rescue_vision_core.py)"
        ENGINE --> YOLO[YOLO11 Pose 추론<br>Keypoints & Bboxes]
        
        YOLO --> VIS{Visibility Classification}
        VIS -- FULL_BODY --> C1(Extract: Shoulder & Hip Center)
        VIS -- UPPER_BODY --> C2(Extract: Shoulder Center)
        VIS -- PARTIAL --> C3(Extract: BBox Center)

        C1 --> POS{Posture Classification}
        C2 --> POS
        C3 --> POS

        POS --> MOT[Motion Analysis over time<br>NONE / LOW / ACTIVE]
        MOT --> DECIDE{Rules & Time Decision<br>ex: LYING + NONE > 7.0s}
        DECIDE --> RES((Result:<br>CRITICAL / WARNING / CAUTION / NORMAL))
    end

    %% Returning Results
    RES --> ENC[Encode Annotated Frame to JPG]
    RES --> PUB1
    ENC --> PUB2
```

### 🚀 판단 기준 요약 (Evaluation Criteria Summary)

현재 파이썬 코드(`rescue_vision_core.py`)에 하드코딩된 주요 판별 조건절을 알기 쉽게 정리한 표입니다.

| 단계 (Step) | 분류 (Class) | 판단 기준 (Thresholds) |
|---|---|---|
| **가시성<br>(Visibility)** | `FULL_BODY` | 머리, 양 어깨, 양 골반 관절의 신뢰도가 모두 0.5 이상일 때 |
| | `UPPER_BODY` | 골반은 잘렸지만 머리와 양 어깨까지만 확실히 렌더링될 때 |
| | `PARTIAL` | 머리와 어깨마저 불확실하게 짤리거나 겹쳐서 안 보일 때 |
| **자세<br>(Posture)** | `LYING` | (전신 전용) Bbox 가로가 세로의 2배 이상 길거나 척추(어깨-골반) 각도가 45도 넘게 기움 |
| | `COLLAPSED` | 허리/어깨선이 45도 이상 심하게 무너졌거나, 고개 꺾임(drop_ratio)이 0.5~0.6 이상일 때 |
| | `LEANING` | 어깨가 15도 이상 기울었거나 고개가 30% 이상 꺾여 불안정한 자세를 아슬아슬하게 유지 |
| | `NORMAL` | 위 항목에 해당하지 않는 수직에 가까운 안정적인 서기/앉기 자세 |
| **움직임<br>(Motion)** | `ACTIVE` | 부드러운 움직임(smooth)이나 상체 움직임(upper) 픽셀 평균이 기준치를 초과함 |
| | `LOCAL_ONLY` | 상체/팔은 발버둥 치지만 코어(어깨/골반 중심)가 꼼짝없이 고정된 경우 (깔림 의심) |
| | `LOW` | 작은 떨림 수준만 감지되는 미세한 움직임 |
| | `NONE` | 움직임 제로. 의식을 잃었거나 기절 의심 |
| **최종 판정<br>(Emergency)**| `CRITICAL` 🚨 | (전신) 누움/붕괴 자세 + 움직임 `NONE` 상태가 **7.0초 이상** 변함없이 지속 |
| | `WARNING` 🔴 | (전신) 누움/붕괴 + 미세 움직임 `LOW/NONE` 상태가 **5.5초 이상** 지속 |
| | `CAUTION` 🟡 | 몸이 기울었거나 아무 자세에서나 움직임이 `LOW/NONE`으로 **4.5초 이상** 지속 |
| | `NORMAL` 🟢 | 그 외의 모든 안전/활동 반경 내의 판정들 |


---

### 📡 ROS2 Topics & 🧭 Code Line References

#### 1. ROS2 Topics (`rescue_vision_node.py`)

| Topic Name Parameter | Default Value | Message Type | I/O |
|---|---|---|---|
| `input_image_topic` | `/camera/image_raw/compressed` | `sensor_msgs/CompressedImage` | **Subscribe** |
| `emergency_level_topic` | `/robot6/emergency_level` | `std_msgs/String` | **Publish** |
| `image_result_topic` | `/robot6/image_result/compressed` | `sensor_msgs/CompressedImage` | **Publish** |

#### 2. 주요 로직 라인 넘버 (Line Numbers)

> **`rescue_vision_node.py` (ROS2 Wrapper)**
- `__init__` (파라미터/Pub/Sub 선언): Line 47~103
- `image_callback` (메인 루프): Line 148~180
- 디코딩/인코딩 유틸 (`_decode_compressed_image` 등): Line 108~143

> **`rescue_vision_core.py` (Core Engine)**
- `_classify_visibility` (가시성 판단): Line 319~361
- `_classify_posture` (자세 수학적 계산): Line 366~497
- `_classify_motion` (시간/프레임 비교 움직임 계산): Line 557~574
- `_decide` (최종 위급수준 시간적 판정): Line 607~683
- `analyze_frame_with_results` (YOLO + 위 로직 종합 오케스트레이터): Line 776~910
